# Run the FSO assignment directly from GitHub

This notebook clones the repository, runs the full simulation (`code/run_all.py`) and shows the figures and the main numbers.

Run the cells from top to bottom (**Runtime → Run all**). No installation is needed: Google Colab already has NumPy, SciPy and Matplotlib.

## 1. Get the code (clone the first time, update afterwards)

In [ ]:
import os
%cd /content
REPO = "Assignment-free-space-optics-with-turbulence-and-pointing-errors"
if os.path.exists(REPO):
    %cd {REPO}
    !git pull
else:
    !git clone https://github.com/sweety-w-sys/Assignment-free-space-optics-with-turbulence-and-pointing-errors.git
    %cd {REPO}
!ls

## 2. Run the simulation

* Full run: about 10–15 minutes (the interleaver study takes most of the time).
* Set `QUICK = True` for a faster check with 10× fewer Monte Carlo bits.

In [ ]:
QUICK = False
%cd /content/Assignment-free-space-optics-with-turbulence-and-pointing-errors/code
if QUICK:
    !python run_all.py --quick
else:
    !python run_all.py

## 3. Show all figures

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('/content/Assignment-free-space-optics-with-turbulence-and-pointing-errors/figures/*.png')):
    print(f.split('/')[-1])
    display(Image(f))

## 4. Main numbers

In [ ]:
import json
r = json.load(open('/content/Assignment-free-space-optics-with-turbulence-and-pointing-errors/results/results.json'))

print('Nine operating points (Task 3)')
print(f"{'sigma_s':>8} {'sigma_R2':>9} {'kappa':>7} {'area':>8} {'D17 [dB]':>9} {'D18 [dB]':>9}")
for p in r['nine_points']:
    print(f"{p['sigma_s']:8.2f} {p['sigma_R2']:9.1f} {p['kappa']:7.2f} {p['area_1e6']:8.4f} {p['D17']:9.4f} {p['D18']:9.3f}")

print('\nFig. 2 operating points (Task 2)')
for name, c in r['curves'].items():
    print(f"{name:7s} P16 = {c['P16']:6.2f} dBm   D17 = {c['D17']:.3f} dB   D18 = {c['D18']:.3f} dB   MC - exact = {c['Dmc']:+.3f} dB")

print('\nOutage (Task 4)')
for name in ['green', 'yellow', 'pink']:
    o = r['outage'][name]
    print(f"{name:7s} P(avg BER = HD-FEC) = {o['P_avg']:6.2f} dBm   P(Pout = 1e-3) = {o['P_out_0.001']:6.2f} dBm   penalty = {o['penalty_1e-3']:.1f} dB")

## 5. (Optional) Rebuild the report PDF with the new figures

Installs LaTeX the first time (3–5 minutes), then compiles `report/report.tex`.

In [ ]:
!apt-get -qq install -y texlive-latex-extra texlive-fonts-recommended texlive-science lmodern > /dev/null
%cd /content/Assignment-free-space-optics-with-turbulence-and-pointing-errors/report
!pdflatex -interaction=nonstopmode report.tex > /dev/null
!pdflatex -interaction=nonstopmode report.tex > /dev/null
!ls -la report.pdf

## 6. (Optional, owner only) Push the new results to GitHub

Needs a GitHub token saved in Colab **Secrets** (key icon on the left) under the name `GH_TOKEN`, with *Notebook access* switched on.

In [ ]:
from google.colab import userdata
t = userdata.get('GH_TOKEN')
%cd /content/Assignment-free-space-optics-with-turbulence-and-pointing-errors
!git config user.name "sweety-w-sys"
!git config user.email "148787313+sweety-w-sys@users.noreply.github.com"
!git add figures results report/report.pdf
!git commit -m "Update results from Colab run"
!git push https://{t}@github.com/sweety-w-sys/Assignment-free-space-optics-with-turbulence-and-pointing-errors.git main
!git fetch
!git status

## 7. Download the results

In [ ]:
!cd /content/Assignment-free-space-optics-with-turbulence-and-pointing-errors && zip -qr /content/fso_results.zip figures results report/report.pdf
from google.colab import files
files.download('/content/fso_results.zip')